# 1. 필요한 라이브러리 임포트

In [1]:
import pandas as pd
import numpy as np
import re
import os
import networkx as nx
import ollama
from konlpy.tag import Okt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rouge_score import rouge_scorer
import torch
from bert_score import BERTScorer
import warnings
warnings.filterwarnings('ignore')

# 2. 한국어 불용어 데이터 로드
`stopwords-ko.txt`에서 불용어 목록을 가져와 변수로 저장합니다.

In [2]:
with open('stopwords-ko.txt', 'r', encoding='utf-8') as f:
    stopwords = set(f.read().splitlines())
print(f"로드된 한국어 불용어 개수: {len(stopwords)}")

로드된 한국어 불용어 개수: 595


In [3]:
# 하드코딩한 불용어
# 텍스트 구조를 해치는 불필요한 부사나 접속사, 꼬리말, 뉴스 매체명 등을 여기에 추가하세요.
# 예시: "결론적으로", "사실상", "등", "이", "그", "저", "설명이다", "전했다", "말했다"
custom_stopwords = [
    '기자', '뉴스', '연구팀', '교수', '박사', '한국과학기술원', '카이스트', 'KAIST',
    '이번', '지난', '최근', '통해', '관련', '기술', '내용', '사실', '결과', '설명',
    '연구진', '벤처비트', '칼럼', '전문가', '관계자', '대표', '총장', '따르면', 
    '현지시간', '통신', '업계', '이날', '가운데', '가장', '크게', '대한', '위해'
]

stopwords.update(custom_stopwords)
print(f"불용어 추가 후 개수: {len(stopwords)}")

불용어 추가 후 개수: 629


In [4]:
import re

def clean_news_text(text):
    if not isinstance(text, str): return text
    
    # 1. 괄호 안의 소속, 부연설명, 한자, 기자 이름 제거
    text = re.sub(r'\([^{|가-힣a-zA-Z0-9]*[가-힣a-zA-Z0-9,\s]*\)', '', text)
    
    # 2. 날짜, 시간 및 특정 매체명 제거 (단, 서술어나 조사를 건드리지 않고 날짜 덩어리만 제거)
    text = re.sub(r'\d+일(\(현지시간\))?\s*', '', text)
    
    # 3. 보도 상투어 패턴 변환 (가장 뒤에 붙는 쓸데없는 '설명이다' 류만 가볍게 정제)
    text = re.sub(r'([가-힣]+)다는\s+(설명이다|방침이다|계획이다|전언이다|전망이다|지적이다)\.', r'\1다.', text)
    text = re.sub(r'([가-힣]+)라는\s+(설명이다|분석이다|평가다)\.', r'\1다.', text)
    
    # 4. 불필요한 기호 및 이중 공백 정리
    text = re.sub(r'["“’‘”\']', '', text)
    text = re.sub(r'[ \t]+', ' ', text)
    
    # 공백 제거 후 어색해진 부분 정리
    text = text.replace(' .', '.')
    text = text.replace(' ,', ',')
    
    return text.strip()

# 3. TF-IDF 및 Ollama 요약기 클래스 정의
기존 `backend-server/summarizer.py`의 구조를 가져와, TF-IDF 기반으로 먼저 요약문장을 선택하고, 그 선택된 문장들에서 불용어(`stopwords-ko.txt`)를 제거하는 방식(`remove_stopwords`)으로 수정합니다.

In [12]:
class SummarizerWithStopwords:
    def __init__(self, stopwords_set, ollama_model="gemma3:27b"):
        self.okt = Okt()
        self.vectorizer = TfidfVectorizer()
        self.ollama_model = os.getenv("OLLAMA_MODEL", ollama_model)
        self.stopwords = stopwords_set

    def get_nouns(self, text):
        """명사 추출 및 전처리 (기본 TF-IDF 용)"""
        if not isinstance(text, str): return ""
        nouns = self.okt.nouns(text)
        return " ".join([n for n in nouns if len(n) > 1])

    def split_sentences(self, text):
        """문장 분리"""
        sentences = re.split(r'(?<=[.!?])\s+', str(text).strip())
        return [s for s in sentences if len(s.strip()) > 5]
        
    def remove_stopwords(self, text):
        """생성된 요약문에서 불용어 단어들 제거 (줄바꿈과 구두점 보존)"""
        lines = text.split('\n')
        processed_lines = []
        for line in lines:
            if not line.strip():
                processed_lines.append(line)
                continue
            
            words = line.split()
            filtered_words = []
            for w in words:
                clean_w = re.sub(r'[^가-힣a-zA-Z0-9]', '', w)
                if clean_w not in self.stopwords and w not in self.stopwords:
                    filtered_words.append(w)
                    
            processed_lines.append(" ".join(filtered_words))
            
        return "\n".join(processed_lines)

    def tfidf_summary(self, text, top_n=5):
        """TF-IDF 기반 요약 & 정제 전/후 두 가지 결과 반환"""
        sentences = self.split_sentences(text)
        
        # 1. 원본 TF-IDF 요약 (정제 처리 안 함)
        if len(sentences) <= top_n: 
            raw_summary = " ".join(sentences)
        else:
            sent_nouns = [self.get_nouns(s) for s in sentences]
            try:
                tfidf_matrix = self.vectorizer.fit_transform(sent_nouns)
                scores = np.array(tfidf_matrix.sum(axis=1)).flatten()
                top_indices = scores.argsort()[-top_n:][::-1]
                top_indices.sort()
                summary_sentences = [sentences[i] for i in top_indices]
                raw_summary = "\n\n".join(summary_sentences)
            except:
                raw_summary = "\n\n".join(sentences[:top_n])
                
        # 2. 후처리 진행된 요약
        processed_summary = raw_summary
        if 'clean_news_text' in globals():
            processed_summary = clean_news_text(processed_summary)
            
        processed_summary = self.remove_stopwords(processed_summary)
        
        # 두 버전을 모두 리턴합니다 (원본, 처리본)
        return raw_summary, processed_summary

    def ollama_summary(self, text):
        """Ollama AI 요약"""
        prompt = f"""다음 기사를 정확히 5개의 문장으로 요약하라.

                    [출력 규칙]
                    - 반드시 5줄만 출력한다.
                    - 각 줄은 하나의 완전한 문장만 포함한다.
                    - 줄 구분은 단일 개행 문자(\n)만 사용한다.
                    - 줄 사이에 빈 줄을 절대 넣지 않는다.
                    - 번호, 기호, 불릿, 설명을 절대 포함하지 않는다.
                    - 첫 줄 앞과 마지막 줄 뒤에 공백이나 개행을 넣지 않는다.
                    - 출력은 오직 요약문 5줄만 작성한다.

                    {text}"""
        try:
            response = ollama.chat(model=self.ollama_model, messages=[
                {'role': 'user', 'content': prompt},
            ])
            content = response['message']['content']
            formatted_content = re.sub(r'(?<=[.!?])\s+', '\n\n', content)
            return formatted_content
        except Exception as e:
            return f"Ollama Error: {str(e)}"

# 4. 테스트 텍스트로 요약 진행 및 결과 출력

In [27]:
# 예시 텍스트 입력 부분
sample_text = """
한국과학기술원(KAIST, 총장 이광형)은 한동수 전기및전자공학부 교수 연구팀이 데이터센터 이외의 개인 PC 및 모바일 등의 인프라를 활용해 LLM 추론(서비스) 비용을 크게 낮추는 기술 ‘ 스펙엣지(SpecEdge) ’를 개발했다고 28일 밝혔다. 스펙엣지는 데이터센터 GPU와 개인 PC·소형 서버 등의 엣지 GPU가 역할을 나눠 LLM 추론 인프라를 구성하는 기술이다. 기존 데이터센터 GPU만 사용하는 방식에 비해, AI가 하나의 토큰을 생성하는 데 소요되는 비용을 67.6% 절감할 수 있다고 전했다. 연구팀은 이를 위해 ‘추측적 디코딩(Speculative Decoding)’이라는 방법을 활용했다. 엣지 GPU에 배치된 소형언어모델(sLM)이 확률이 높은 토큰 시퀀스를 빠르게 생성하면, 데이터센터의 대형언어모델(LLM)이 이를 일괄 검증하는 방식이다. 엣지 GPU가 서버의 응답을 기다리지 않고 계속 단어를 만들어내기 때문에, 추론속도와 인프라 효율을 동시에 높일 수 있다는 설명이다. 특히, 데이터센터 GPU에서만 추측적 디코딩을 수행하는 방식과 비교하면 비용 효율은 1.91배, 서버 처리량은 2.22배 향상한 것으로 나타났다. 일반적인 인터넷 속도에서도 문제 없이 작동해, 별도 특수 네트워크 환경이 없어도 실제 서비스에 바로 적용이 가능한 수준이라고 전했다. 또, 메인 서버는 여러 엣지 GPU의 검증 요청을 효율적으로 처리하도록 설계됐기 때문에, GPU 유휴 시간 없이 더 많은 요청을 동시 처리할 수 있다는 설명이다. 데이터센터 자원을 효율적으로 활용할 수 있는 LLM 서빙 인프라 구조를 구현한 것이다. 이에 따라 데이터센터에 집중돼 있던 LLM 연산을 엣지로 분산, AI 서비스의 핵심인 인프라 비용은 줄이면서 접근성은 높일 수 있도록 가능성을 제시했다는 결론이다. 앞으로 스마트폰, 개인용 컴퓨터, NPU 등 엣지 기기로 기술을 확장, 많은 사용자들이 고품질 AI서비스를 활용할 수 있을 것으로 전망했다. 한동수 KAIST 교수는 “데이터센터를 넘어서 사용자 주변에 있는 엣지 자원까지 LLM 인프라로 활용하는 것이 목표”라고 말했다.
"""

summarizer = SummarizerWithStopwords(stopwords)

raw_tfidf, processed_tfidf = summarizer.tfidf_summary(sample_text, top_n=5)

print("--- [1. TF-IDF 요약 결과 (정제 처리 전 원본)] ---")
print(raw_tfidf)

print("\n--- [2. TF-IDF 요약 결과 (정제 및 불용어 제거 후)] ---")
print(processed_tfidf)

print("\n--- [3. Ollama (Ground Truth) 요약 결과] ---")
ollama_result = summarizer.ollama_summary(sample_text)
print(ollama_result)

--- [1. TF-IDF 요약 결과 (정제 처리 전 원본)] ---
한국과학기술원(KAIST, 총장 이광형)은 한동수 전기및전자공학부 교수 연구팀이 데이터센터 이외의 개인 PC 및 모바일 등의 인프라를 활용해 LLM 추론(서비스) 비용을 크게 낮추는 기술 ‘ 스펙엣지(SpecEdge) ’를 개발했다고 28일 밝혔다.

엣지 GPU에 배치된 소형언어모델(sLM)이 확률이 높은 토큰 시퀀스를 빠르게 생성하면, 데이터센터의 대형언어모델(LLM)이 이를 일괄 검증하는 방식이다.

엣지 GPU가 서버의 응답을 기다리지 않고 계속 단어를 만들어내기 때문에, 추론속도와 인프라 효율을 동시에 높일 수 있다는 설명이다.

일반적인 인터넷 속도에서도 문제 없이 작동해, 별도 특수 네트워크 환경이 없어도 실제 서비스에 바로 적용이 가능한 수준이라고 전했다.

이에 따라 데이터센터에 집중돼 있던 LLM 연산을 엣지로 분산, AI 서비스의 핵심인 인프라 비용은 줄이면서 접근성은 높일 수 있도록 가능성을 제시했다는 결론이다.

--- [2. TF-IDF 요약 결과 (정제 및 불용어 제거 후)] ---
한국과학기술원은 한동수 전기및전자공학부 연구팀이 데이터센터 이외의 개인 PC 모바일 등의 인프라를 활용해 LLM 추론 비용을 낮추는 스펙엣지 개발했다고 밝혔다.

엣지 GPU에 배치된 소형언어모델이 확률이 높은 토큰 시퀀스를 빠르게 생성하면, 데이터센터의 대형언어모델이 이를 일괄 검증하는 방식이다.

엣지 GPU가 서버의 응답을 기다리지 않고 계속 단어를 만들어내기 추론속도와 인프라 효율을 높일 수

일반적인 인터넷 속도에서도 문제 없이 작동해, 별도 특수 네트워크 환경이 없어도 실제 서비스에 적용이 가능한 수준이라고 전했다.

이에 데이터센터에 집중돼 있던 LLM 연산을 엣지로 분산, AI 서비스의 핵심인 인프라 비용은 줄이면서 접근성은 높일 수 있도록 가능성을 제시했다는 결론이다.

--- [3. Ollama (Ground Truth) 요약 결과] ---
KAIST 연구팀은 개인 P

# 5. ROUGE & BERT 점수 측정
Ollama가 생성한 요약을 `[Reference]` 로, TF-IDF 기반으로 생성한 요약을 `[Candidate]` 로 설정하여 ROUGE 및 BERTScore를 측정합니다.

In [28]:
def evaluate_scores(reference, candidate, label):
    print(f"\n=== 평가 결과 ({label}) ===")
    
    # 1. ROUGE Score
    rouge_evaluator = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    rouge_scores = rouge_evaluator.score(reference, candidate)
    print("[ROUGE Scores]")
    print(f"- ROUGE-1 F1: {rouge_scores['rouge1'].fmeasure:.4f}")
    print(f"- ROUGE-2 F1: {rouge_scores['rouge2'].fmeasure:.4f}")
    print(f"- ROUGE-L F1: {rouge_scores['rougeL'].fmeasure:.4f}")
    
    # 2. BERTScore
    try:
        device = "cuda" if torch.cuda.is_available() else "cpu"
        bert_scorer_model = BERTScorer(lang="ko", rescale_with_baseline=False, device=device)
        P, R, F1 = bert_scorer_model.score([candidate], [reference])
        print("\n[BERT Scores]")
        print(f"- BERT-P: {P.item():.4f}")
        print(f"- BERT-R: {R.item():.4f}")
        print(f"- BERT-F1: {F1.item():.4f}")
    except Exception as e:
        print(f"\nBERTScore 계산 실패: {e}")

# 정제 전/후 각각 스코어 평가
evaluate_scores(reference=ollama_result, candidate=raw_tfidf, label="정제 처리 전 TF-IDF vs Ollama")
evaluate_scores(reference=ollama_result, candidate=processed_tfidf, label="정제 처리 후 TF-IDF vs Ollama")


=== 평가 결과 (정제 처리 전 TF-IDF vs Ollama) ===
[ROUGE Scores]
- ROUGE-1 F1: 0.6087
- ROUGE-2 F1: 0.4762
- ROUGE-L F1: 0.6087

[BERT Scores]
- BERT-P: 0.7490
- BERT-R: 0.7897
- BERT-F1: 0.7688

=== 평가 결과 (정제 처리 후 TF-IDF vs Ollama) ===
[ROUGE Scores]
- ROUGE-1 F1: 0.6667
- ROUGE-2 F1: 0.6250
- ROUGE-L F1: 0.6667

[BERT Scores]
- BERT-P: 0.7651
- BERT-R: 0.7816
- BERT-F1: 0.7732
